In [1]:
!pip install -U "langchain>=1.0.0" "langchain-openai>=0.2.0" "langchain-community>=0.3.0" \
               "langchain-text-splitters>=0.2.0" "tiktoken>=0.7.0" "chromadb>=0.5.5" \
               "pymupdf>=1.24.0" "gradio>=4.44.0" "requests>=2.31.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 k

In [2]:
import os
import re
import io
import sys
import json
import requests
from typing import Dict, List, Optional, Tuple
from contextlib import redirect_stdout

# LangChain 1.0 계열 임포트
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.tools import create_retriever_tool
from langchain_core.prompts import PromptTemplate

import gradio as gr

In [3]:
os.environ["OPENAI_API_KEY"] = "여러분의 키 값"

In [4]:
urls = [
   "https://raw.githubusercontent.com/ai-agent-kr/agent-tutorial/main/Ch02/ict_japan_2024.pdf",
   "https://raw.githubusercontent.com/ai-agent-kr/agent-tutorial/main/Ch02/ict_usa_2024.pdf",
   "https://raw.githubusercontent.com/ai-agent-kr/agent-tutorial/main/Ch02/blockchain_usa_2025.pdf"
]

# 각 파일 다운로드
for url in urls:
   filename = url.split("/")[-1]  # URL에서 파일명 추출
   response = requests.get(url)

   with open(filename, "wb") as f:
       f.write(response.content)
   print(f"{filename} 다운로드 완료")

ict_japan_2024.pdf 다운로드 완료
ict_usa_2024.pdf 다운로드 완료
blockchain_usa_2025.pdf 다운로드 완료


In [5]:
embd = OpenAIEmbeddings()

In [6]:
def create_pdf_retriever(
    pdf_path: str,  # PDF 파일 경로
    persist_directory: str,  # 벡터 스토어 저장 경로
    embedding_model: OpenAIEmbeddings,  # OpenAIEmbeddings 임베딩 모델
    chunk_size: int = 512,  # 청크 크기 기본값 512
    chunk_overlap: int = 0  # 청크 오버랩 크기 기본값 0
):
    loader = PyMuPDFLoader(pdf_path)
    data = loader.load()

    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    doc_splits = text_splitter.split_documents(data)

    vectorstore = Chroma.from_documents(
        documents=doc_splits,
        embedding=embedding_model,
        persist_directory=persist_directory,
    )
    return vectorstore.as_retriever()

In [7]:
retriever_ict_japan = create_pdf_retriever(
    pdf_path="ict_japan_2024.pdf",
    persist_directory="db_ict_policy_japan_2024",
    embedding_model=embd
)

# 미국 ICT 정책 데이터베이스 생성
retriever_ict_usa = create_pdf_retriever(
    pdf_path="ict_usa_2024.pdf",
    persist_directory="db_ict_policy_usa_2024",
    embedding_model=embd
)

# 미국 블록체인 동향 데이터베이스 생성
retriever_blockchain_usa = create_pdf_retriever(
    pdf_path="blockchain_usa_2025.pdf",
    persist_directory="db_blockchain_usa_2025",
    embedding_model=embd
)

In [8]:
ict_japan_engine = create_retriever_tool(
    retriever=retriever_ict_japan,
    name="japan_ict_trend_searcher",
    description="일본의 ICT 산업의 시장동향 정보를 제공합니다."
)

ict_usa_engine = create_retriever_tool(
    retriever=retriever_ict_usa,
    name="usa_ict_trend_searcher",
    description="미국의 ICT 산업의 시장동향 정보를 제공합니다."
)

blockchain_usa_engine = create_retriever_tool(
    retriever=retriever_blockchain_usa,
    name="usa_blockchain_trend_searcher",
    description="미국의 블록체인 산업의 동향 정보를 제공합니다."
)

In [9]:
tools = [ict_japan_engine, ict_usa_engine, blockchain_usa_engine]

In [10]:
tool_map: Dict[str, object] = {t.name: t for t in tools}

In [11]:
react_template = '''다음 질문에 최선을 다해 답변하세요. 당신은 다음 도구들에 접근할 수 있습니다:

{tools}

다음 형식을 사용하세요:

Question: 답변해야 하는 입력 질문
Thought: 무엇을 할지 항상 생각하세요
Action: 취해야 할 행동, [{tool_names}] 중 하나여야 합니다. 리스트에 있는 도구 중 1개를 택하십시오.
Action Input: 행동에 대한 입력값
Observation: 행동의 결과
... (이 Thought/Action/Action Input/Observation의 과정이 N번 반복될 수 있습니다)
Thought: 이제 최종 답변을 알겠습니다
Final Answer: 원래 입력된 질문에 대한 최종 답변 (한글로 작성하십시오.)

## 추가적인 주의사항
- 반드시 [Thought -> Action -> Action Input -> Observation] 순서를 준수하십시오. 항상 Action 전에는 Thought가 먼저 나와야 합니다.
- 최종 답변에는 최대한 많은 내용을 포함하십시오.
- 한 번의 검색으로 해결되지 않을 것 같다면 문제를 분할하여 푸는 것도 고려하십시오.
- 정보가 취합되었다면 불필요하게 사이클을 반복하지 마십시오.
- 묻지 않은 정보를 찾으려고 도구를 사용하지 마십시오.

시작하세요!

Question: {input}
{agent_scratchpad}'''

In [12]:
prompt = PromptTemplate.from_template(react_template)

In [13]:
def _format_tools_for_prompt(ts: List[object]) -> Tuple[str, str]:
    lines, names = [], []
    for t in ts:
        names.append(t.name)
        desc = getattr(t, "description", "")
        lines.append(f"{t.name}: {desc}")
    return "\n".join(lines), ", ".join(names)

def _render_prompt(user_input: str, scratchpad: str) -> str:
    tools_str, tool_names = _format_tools_for_prompt(tools)
    return prompt.format(
        tools=tools_str,
        tool_names=tool_names,
        input=user_input,
        agent_scratchpad=scratchpad,
    )

In [14]:
llm = ChatOpenAI(model="gpt-4.1", temperature=0)

In [15]:
# =========================
# ReAct 파서 및 실행 루프
# =========================
ACTION_RE = re.compile(r"^Action\s*:\s*(?P<tool>.+?)\s*$", re.MULTILINE)
ACTION_INPUT_RE = re.compile(r"^Action Input\s*:\s*(?P<input>.+?)\s*$", re.MULTILINE)
FINAL_ANSWER_RE = re.compile(r"Final Answer\s*:\s*(?P<final>[\s\S]+)$", re.IGNORECASE)

def _parse_action_and_input(text: str) -> Tuple[Optional[str], Optional[str]]:
    m_act = ACTION_RE.search(text)
    m_in = ACTION_INPUT_RE.search(text)
    if m_act and m_in:
        tool = m_act.group("tool").strip()
        action_input = m_in.group("input").strip()
        action_input_first_line = action_input.splitlines()[0].strip()
        return tool, action_input_first_line

    m_final = FINAL_ANSWER_RE.search(text)
    if m_final:
        return "__FINAL__", m_final.group("final").strip()

    return None, None

def _observation_to_text(observation_obj) -> str:
    if isinstance(observation_obj, list):
        # Document 리스트일 수 있음
        def doc_to_str(d):
            try:
                meta = getattr(d, "metadata", {}) or {}
                src = meta.get("source") or meta.get("file_path") or ""
                txt = getattr(d, "page_content", "")
                if len(txt) > 500:
                    txt = txt[:500] + "..."
                return f"[source={src}] {txt}"
            except Exception:
                return str(d)
        return "\n".join(doc_to_str(d) for d in observation_obj[:5])
    return str(observation_obj)

STOP_SEQ = ["\nObservation:"]  # LLM이 Observation 내용을 쓰기 직전에 출력이 끊기게 함

def run_react(user_input: str, max_iters: int = 8) -> Dict[str, str]:
    scratchpad = ""
    for _ in range(max_iters):
        rendered = _render_prompt(user_input, scratchpad)
        resp = llm.invoke(rendered, stop=STOP_SEQ)
        text = resp.content if hasattr(resp, "content") else str(resp)

        tool, action_input = _parse_action_and_input(text)
        if tool is None:
            hint = "\n[파싱안내] 형식을 엄격히 따르세요. 반드시 'Action:'와 'Action Input:'을 한 줄씩 제공하십시오.\n"
            scratchpad += f"{text}\n{hint}"
            continue

        if tool == "__FINAL__":
            final_answer = action_input
            return {"output": final_answer, "log": scratchpad + "\n" + text}

        if tool not in tool_map:
            observation = f"[에러] 존재하지 않는 도구입니다: {tool}"
            scratchpad += f"{text}\nObservation: {observation}\n"
            continue

        try:
            observation_obj = tool_map[tool].invoke(action_input)
            observation = _observation_to_text(observation_obj)
            scratchpad += f"{text}\nObservation: {observation}\n"
        except Exception as e:
            scratchpad += f"{text}\nObservation: [도구실행오류] {e}\n"

    return {
        "output": "반복 한도를 초과했습니다. 질문을 더 구체화해 주세요.",
        "log": scratchpad,
    }

In [16]:
# =========================
# 단건 질의
# =========================
try:
    q1 = "미국의 블록체인 시장 규모는 어떤가요?"
    out1 = run_react(q1, max_iters=8)
    print("최종 답변:", out1["output"])
    print("\n=== 실행 로그 (Thought / Action / Action Input / Observation) ===\n")
    print(out1["log"])
except Exception as e:
    print(f"단건 질의 중 오류: {e}")

최종 답변: 미국의 블록체인 시장은 2024년 기준 약 99억 달러(약 13조 원) 규모로 추정되며, 2032년에는 약 1조 7,665억 달러(약 2,400조 원)까지 성장할 것으로 전망됩니다. 이는 2024년부터 2032년까지 연평균 92.4%라는 매우 높은 성장률을 기록할 것으로 예상된다는 의미입니다. 이러한 급격한 시장 확대는 금융, 공급망, 의료, 공공부문 등 다양한 산업에서 데이터 보안 강화와 거래 효율성 개선을 위해 블록체인 기술 도입이 빠르게 확산되고 있기 때문입니다. 또한 DeFi(탈중앙화 금융)와 AI(인공지능)와의 결합을 통한 혁신 서비스가 등장하면서, 미국은 블록체인 및 암호화폐 분야에서 글로벌 리더로 자리매김하고 있습니다.

=== 실행 로그 (Thought / Action / Action Input / Observation) ===

Thought: 미국의 블록체인 시장 규모에 대한 최신 동향과 수치를 확인하기 위해 관련 도구를 사용해야 합니다.
Action: usa_blockchain_trend_searcher
Action Input: 미국 블록체인 시장 규모 및 성장 동향
Observation: 미국 — 블록체인

Ⅰ 시장 유망성 
6
2. 시장 규모
n  미국 블록체인 시장, 2024-2032년 연평균 92.4% 성장 전망
- 미국 블록체인 시장은 2024년 99억 달러에서 2032년 1조 7,665억 달러로 약 
178배 확대될 것으로 전망되며, 연평균 성장률 92.4%를 기록할 것으로 예상됨
- 금융, 공급망, 의료, 공공부문 등 다양한 산업에서 데이터 보안 강화와 거래 
효율성 개선을 위한 블록체인 도입이 확산되며 시장 규모가 빠르게 확대될 전망임

ICT 해외진출 전략가이드
3
SUMMARY
데이터센터 시장 규모
§
미국 블록체인 시장의 연평균 92.4% 성장률 및 
2032년 1조 7,665억 달러 규모 도달 전망
§
금융·공급망·의료 분야의 데이터 
보안 및 거래 
효율화로 수요 증가
§
DeFi 확산과 AI 결합으로 금

In [17]:
# =========================
# 멀티 쿼리
# =========================
try:
    q2 = "미국과 일본의 ICT 주요 정책의 공통점과 차이점을 설명해주고 미국의 블록체인 스타트업 지원 제도를 정리해 주세요."
    out2 = run_react(q2, max_iters=10)
    print("최종 답변:", out2["output"])
    print("\n=== 실행 로그 (Thought / Action / Action Input / Observation) ===\n")
    print(out2["log"])
except Exception as e:
    print(f"멀티 쿼리 중 오류: {e}")

최종 답변: 미국과 일본의 ICT 주요 정책의 공통점과 차이점, 그리고 미국의 블록체인 스타트업 지원 제도에 대해 아래와 같이 정리합니다.

---

### 1. 미국과 일본의 ICT 주요 정책의 공통점

- **반도체 및 디지털 산업 강화**: 양국 모두 반도체 산업을 국가 전략산업으로 지정하고, 연구개발(R&D) 및 제조 지원을 강화하고 있습니다. 미국은 CHIPS 법안을 통해 대규모 예산을 투입하고 있으며, 일본도 ‘반도체·디지털 산업전략’을 통해 국내 생산 촉진과 디지털 인프라 강화를 추진합니다.
- **차세대 기술 선도**: 인공지능(AI), 6G, 양자컴퓨팅 등 첨단 ICT 분야에서 글로벌 경쟁력 확보를 목표로 정책을 추진합니다. 미국은 AI 챗봇, 6G, 양자컴퓨팅, 의료 AI 등 다양한 분야에 집중하고 있고, 일본도 6G, 양자컴퓨터, Web3 등 신기술 개발에 힘쓰고 있습니다.
- **사이버보안 강화**: 양국 모두 사이버보안 역량 강화를 중요한 정책 목표로 삼고 있습니다.

---

### 2. 미국과 일본의 ICT 주요 정책의 차이점

- **혁신지수 및 글로벌 위상**: 미국은 글로벌 혁신지수 3위로, ‘지식 및 기술 생산’에서 강점을 보입니다. 일본은 13위로, ‘인프라’와 ‘지식 및 기술 생산’은 강점이나 ‘창조적 생산’은 약점으로 평가됩니다.
- **정책 추진 방식**: 미국은 민간 주도의 혁신과 정부의 대규모 재정 지원(예: CHIPS Act) 및 국가 간 기술 협력(예: 일본과의 양자컴퓨팅 협력)에 적극적입니다. 일본은 정부 주도의 산업경쟁력 강화법 개정, 전략 분야 집중 지원 등 정책적 지원과 규제 완화에 중점을 둡니다.
- **디지털 사회 실현 목표**: 일본은 ‘고도의 디지털 사회 실현’을 명확한 정책 목표로 내세우며, 행정 업무에 생성형 AI 도입 등 정부 디지털화에 적극적입니다. 미국은 민간 혁신과 글로벌 시장 주도에 더 초점을 둡니다.
- **시장 및 인프라 현황**: 미국은 인터넷 보급률(97.1%)과 혁신 역량이 매

In [ ]:
# ==== Gradio 4.x: Chatbot은 messages 형식을 기대 ====

class ConversationManager:
    def __init__(self):
        self.chat_history = []     # [(user_text, assistant_text), ...] 내부 보관은 유지
        self.execution_logs = []

    def process_message(self, message: str):
        self.execution_logs = []
        try:
            context_str = ""
            if self.chat_history:
                context_str = "이전 대화 내용:\n"
                for i, (q, a) in enumerate(self.chat_history):
                    context_str += f"질문 {i+1}: {q}\n답변 {i+1}: {a}\n"
                context_str += "\n위 대화 맥락을 고려해서 다음 질문에 답변해주세요.\n\n"

            full_message = context_str + message

            f = io.StringIO()
            with redirect_stdout(f):
                result = run_react(full_message, max_iters=10)
                response = result["output"]
                exec_log = result.get("log", "")

            execution_log = f.getvalue() + "\n" + exec_log
            self.execution_logs.append(execution_log)

            # 내부 기록은 기존대로 유지
            self.chat_history.append((message, response))
            return response, execution_log
        except Exception as e:
            error_message = f"오류가 발생했습니다: {str(e)}"
            self.chat_history.append((message, error_message))
            return error_message, f"실행 중 오류: {str(e)}"

    def clear_history(self):
        self.chat_history = []
        self.execution_logs = []
        return []

conversation_manager = ConversationManager()

def chat_with_agent(message, history):
    try:
        response, execution_log = conversation_manager.process_message(message)
    except Exception as e:
        err_msg = f"[오류] {type(e).__name__}: {e}"
        print(err_msg)  # 콘솔에 오류 출력
        response = err_msg
        execution_log = err_msg

    history = (history or []) + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response},
    ]
    return "", history, execution_log

def clear_chat_history():
    conversation_manager.clear_history()
    return [], ""   # 빈 messages, 빈 로그

with gr.Blocks() as demo:
    gr.HTML("<style>footer{visibility:hidden}</style>")
    gr.Markdown("# ReACT 에이전트")
    gr.Markdown("일본과 미국의 ICT 정책 또는 미국의 블록체인 동향에 관한 질문을 해보세요. 대화 컨텍스트를 유지합니다.")

    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=500)  # type 인자 없이 사용. 단, messages 형식 데이터를 넘겨야 함.
            msg = gr.Textbox(label="질문을 입력하세요", placeholder="예: 일본의 AI 정책에 대해 알려줘")
            clear = gr.Button("대화 초기화")

        with gr.Column(scale=1):
            logs = gr.Textbox(label="ReACT 실행 로그", lines=25, max_lines=25)

    msg.submit(chat_with_agent, inputs=[msg, chatbot], outputs=[msg, chatbot, logs])
    clear.click(clear_chat_history, inputs=None, outputs=[chatbot, logs])

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1394412642e6907d35.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
